# Predictive Maintenance EDA

This notebook performs exploratory data analysis and profile generation for the AI4I 2020 Predictive Maintenance dataset. It also demonstrates the dataset loader and feature engineering pipeline used by the predictive maintenance module.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import sys
sys.path.append(str(Path.cwd().parents[2]))

from app.data.loaders import AI4IDatasetLoader
from app.data.profiling import DataProfiler
from app.ml.predictive_maintenance.features import add_engineered_features, get_feature_columns
from app.ml.predictive_maintenance.preprocessing import clean_data


In [ ]:
dataset_path = Path("../../../data/raw/ai4i_predictive_maintenance/ai4i.csv")
loader = AI4IDatasetLoader(dataset_path)
df = loader.load()
df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:
", df.isna().sum())

In [ ]:
profile = DataProfiler.profile(df)
profile.dict()

In [ ]:
cleaned = clean_data(df)
engineered = add_engineered_features(cleaned)
features = get_feature_columns()
engineered[features + ["machine_failure"]].describe()

In [ ]:
plt.figure(figsize=(12, 8))
corr = engineered[features + ["machine_failure"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

## Class Balance and Target Distribution

In [ ]:
target_counts = engineered["machine_failure"].value_counts(normalize=True)
print(target_counts)
sns.barplot(x=target_counts.index.astype(str), y=target_counts.values)
plt.title("Machine Failure Class Distribution")
plt.xlabel("machine_failure")
plt.ylabel("proportion")
plt.show()

## Notes
* The feature engineering step adds derived predictors such as temperature delta, wear rate, torque ratio, and temperature ratio.
* The dataset loader validates schema consistency and exposes profiling information for data quality checks.
* The pipeline is designed to support reproducible training and MLflow tracking.